In [ ]:
# [Célula 1: Imports e Leitura]
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
import os




In [ ]:
# Configurações visuais
sns.set_theme(style="whitegrid", palette="dark")

ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")  # Carrega variáveis de ambiente do .env
DIR_DATAPROCESSED = ROOT / os.getenv("DIR_DATAPROCESSED")

df = pd.read_parquet(os.path.join(DIR_DATAPROCESSED, "preprocessado.parquet"))
df = pd.DataFrame(df)
# Lendo o parquet processado
df['periodo'] = pd.to_datetime(df['periodo'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2464341 entries, 0 to 2464340
Data columns (total 17 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   CNPJ                    str           
 1   Código Nacional         int64         
 2   Município               str           
 3   UF                      str           
 4   Modalidade de Cobrança  str           
 5   Tecnologia              str           
 6   Tecnologia Geração      str           
 7   Empresa                 str           
 8   Porte da Prestadora     str           
 9   Tipo de Pessoa          str           
 10  Tipo de Produto         str           
 11  Código IBGE Município   int64         
 12  Grupo Econômico         str           
 13  periodo                 datetime64[us]
 14  acessos                 int32         
 15  ano                     int32         
 16  mes                     int8          
dtypes: datetime64[us](1), int32(2), int64(2), int8(1), str(11

In [ ]:

# [Célula 2: Cálculo do Tamanho do Mercado e MoM (Month-over-Month)]
# Agrupando acessos por mês e empresa
df_crescimento = df.groupby(['periodo', 'Empresa'])['acessos'].sum().unstack(fill_value=0)

# Total de mercado por mês
mercado_total = df_crescimento.sum(axis=1)
mercado_mom = mercado_total.pct_change() * 100

print("--- Evolução do Mercado Total ---")
for mes, total, mom in zip(mercado_total.index, mercado_total, mercado_mom):
    print(f"{mes.date()}: {total:,.0f} acessos | Variação: {mom:.2f}%")

# [Célula 3: Top 10 Crescimento Absoluto e Percentual (Último Mês)]
atual = df_crescimento.iloc[-1]
anterior = df_crescimento.iloc[-2]

# Crescimento Absoluto
cresc_abs = (atual - anterior).nlargest(10)
print("\n--- Top 10 Crescimento Absoluto ---")
print(cresc_abs)

# Crescimento Percentual (Filtrando empresas com mais de 1000 acessos para evitar ruído estatístico)
filtro_vol = atual > 1000
cresc_pct = ((atual[filtro_vol] - anterior[filtro_vol]) / anterior[filtro_vol] * 100).nlargest(10)
print("\n--- Top 10 Crescimento Percentual (Volume > 1k) ---")
print(cresc_pct)

--- Evolução do Mercado Total ---
2021-01-01: 10,494,556 acessos | Variação: nan%
2021-02-01: 10,704,259 acessos | Variação: 2.00%
2021-03-01: 11,013,553 acessos | Variação: 2.89%
2021-04-01: 11,238,002 acessos | Variação: 2.04%
2021-05-01: 11,556,196 acessos | Variação: 2.83%
2021-06-01: 12,218,276 acessos | Variação: 5.73%
2021-07-01: 12,800,698 acessos | Variação: 4.77%
2021-08-01: 13,706,834 acessos | Variação: 7.08%
2021-09-01: 14,253,856 acessos | Variação: 3.99%
2021-10-01: 14,609,025 acessos | Variação: 2.49%
2021-11-01: 14,928,344 acessos | Variação: 2.19%
2021-12-01: 15,269,851 acessos | Variação: 2.29%
2022-01-01: 15,483,186 acessos | Variação: 1.40%
2022-02-01: 15,791,085 acessos | Variação: 1.99%
2022-03-01: 16,107,906 acessos | Variação: 2.01%
2022-04-01: 16,334,033 acessos | Variação: 1.40%
2022-05-01: 15,167,351 acessos | Variação: -7.14%
2022-06-01: 14,525,550 acessos | Variação: -4.23%
2022-07-01: 16,488,860 acessos | Variação: 13.52%
2022-08-01: 16,889,122 acessos | 